# Tag audit mood-based recommender

Audit una tantum per le Fasi 1 e 3: copertura dei tag usati dai segnali mood, sostituzione dei segnali morti e confronto dei vettori mood duplicati.

In [1]:
import os, sys, ast
from collections import Counter

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))
from config import PATH_CLEAN_RECIPES
from models.mood_based import MoodBasedRecommender

df_recipes = pd.read_csv(os.path.join('..', PATH_CLEAN_RECIPES))
total_recipes = len(df_recipes)
total_recipes

15144

## Fase 1 - Copertura dei tag originali

In [2]:
original_signals = {
    'body': {'comfort-food': 2.0, 'hearty': 1.5, 'meat': 1.0},
    'mental': {'comfort-food': 2.0, 'soul-food': 2.0, 'chocolate': 1.5, 'desserts': 1.5},
    'taste': {'rich': 2.0, 'creamy': 2.0, 'cheesy': 1.5},
    'mod_pos': {'fusion': 2.0, 'exotic': 2.0, 'ethic': 1.5},
    'mod_neg': {'traditional': -2.0, 'classic': -2.0, 'old-fashioned': -1.5},
}

def parse_tags(value):
    if isinstance(value, list):
        return {str(t).lower().strip() for t in value}
    if pd.isna(value):
        return set()
    try:
        parsed = ast.literal_eval(value)
    except Exception:
        return set()
    return {str(t).lower().strip() for t in parsed} if isinstance(parsed, list) else set()

tag_sets = df_recipes['tags'].apply(parse_tags)
rows = []
for tag in sorted({tag for signals in original_signals.values() for tag in signals}):
    count = int(tag_sets.apply(lambda tags: tag in tags).sum())
    pct = count / total_recipes * 100
    rows.append({'tag': tag, 'count': count, 'percent': round(pct, 4)})
    if pct < 0.5:
        print(f"WARNING: tag {tag!r} coverage below 0.5% ({count}/{total_recipes}, {pct:.4f}%)")
pd.DataFrame(rows)

               tag  count  percent
0           cheesy      0   0.0000
1        chocolate    650   4.2921
2          classic      0   0.0000
3     comfort-food   2652  17.5119
4           creamy      0   0.0000
5         desserts   2305  15.2205
6            ethic      0   0.0000
7           exotic      0   0.0000
8           fusion      0   0.0000
9           hearty      0   0.0000
10            meat   4366  28.8299
11   old-fashioned      0   0.0000
12            rich      0   0.0000
13       soul-food      0   0.0000
14     traditional      0   0.0000

Tag sotto soglia 0.5%: `cheesy`, `classic`, `creamy`, `ethic`, `exotic`, `fusion`, `hearty`, `old-fashioned`, `rich`, `soul-food`, `traditional`.

## Fase 2 - Copertura dei sostituti usati

In [3]:
replacement_signals = {
    'body': {'comfort-food': 2.0, 'main-dish': 1.2, 'meat': 1.0, 'beef': 0.8, 'pork': 0.8},
    'mental': {'comfort-food': 2.0, 'kid-friendly': 1.2, 'chocolate': 1.5, 'desserts': 1.5},
    'taste': {'cheese': 1.5, 'savory': 1.2, 'sweet': 1.0, 'spicy': 1.0},
    'mod_pos': {'asian': 1.8, 'mexican': 1.6, 'indian': 1.4, 'thai': 1.2, 'chinese': 1.2, 'middle-eastern': 1.2},
    'mod_neg': {'north-american': -1.5, 'american': -1.0, 'comfort-food': -1.0},
}
rows = []
for tag in sorted({tag for signals in replacement_signals.values() for tag in signals}):
    count = int(tag_sets.apply(lambda tags: tag in tags).sum())
    rows.append({'tag': tag, 'count': count, 'percent': round(count / total_recipes * 100, 4)})
pd.DataFrame(rows)

                tag  count  percent
0          american   2700  17.8290
1             asian    856   5.6524
2              beef   1367   9.0267
3            cheese   1347   8.8946
4         chocolate    650   4.2921
5           chinese    161   1.0631
6      comfort-food   2652  17.5119
7          desserts   2305  15.2205
8            indian    132   0.8716
9      kid-friendly   2675  17.6638
10        main-dish   5158  34.0597
11          mexican    564   3.7242
12   middle-eastern    118   0.7792
13             meat   4366  28.8299
14   north-american   4020  26.5452
15             pork   1088   7.1844
16           savory    742   4.8996
17            spicy    715   4.7213
18            sweet    644   4.2525
19             thai     76   0.5018

## Fase 3 - Duplicati prima/dopo

In [4]:
before_duplicate_pairs_1000 = 5261
before_duplicate_vector_groups_1000 = 144

model = MoodBasedRecommender()
model.fit(df_recipes)
rng = np.random.default_rng(42)
idx = rng.choice(len(model.df_mood_vectors), size=1000, replace=False)
rounded = np.round(model.df_mood_vectors[idx], 2)
_, counts = np.unique(rounded, axis=0, return_counts=True)
after_duplicate_pairs_1000 = int(((counts * (counts - 1)) // 2).sum())
after_duplicate_vector_groups_1000 = int((counts > 1).sum())

pd.DataFrame([
    {'version': 'before', 'duplicate_pairs_1000': before_duplicate_pairs_1000, 'duplicate_vector_groups_1000': before_duplicate_vector_groups_1000},
    {'version': 'after', 'duplicate_pairs_1000': after_duplicate_pairs_1000, 'duplicate_vector_groups_1000': after_duplicate_vector_groups_1000},
])

-> Estrazione automatica delle 6 dimensioni del Mood per ogni ricetta...
-> Matrice Mood generata e normalizzata in scala [-5, +5]!
-> Distanza empirica di riferimento (p90): 8.8873


  version  duplicate_pairs_1000  duplicate_vector_groups_1000
0  before                   5261                          144
1   after                      0                            0

In [5]:
print(f"distanza_riferimento_p90 = {model.distanza_riferimento:.4f}")
sample_results = model.recommend(top_k=10)
print('affinita_pct_top10 =', [r['affinita_pct'] for r in sample_results])
print('distanza_geometrica_top10 =', [r['distanza_geometrica'] for r in sample_results])

distanza_riferimento_p90 = 8.8873
affinita_pct_top10 = [84.0, 83.8, 83.8, 83.7, 83.2, 83.1, 83.0, 83.0, 82.9, 82.6]
distanza_geometrica_top10 = [1.4217, 1.4368, 1.4439, 1.4472, 1.4974, 1.5048, 1.5087, 1.5137, 1.5235, 1.5428]


## Verifica A - Price e modification grezzi prima del fix mirato

Questa sezione ricostruisce la formula precedente al fix mirato, per mantenere auditabile il controllo richiesto prima di modificare `models/mood_based.py`.

In [6]:
import plotly.express as px

raw_before = df_recipes.copy()
raw_before['tags'] = raw_before['tags'].apply(parse_tags)
raw_before['ingredients'] = raw_before['ingredients'].apply(
    lambda value: set(ast.literal_eval(value)) if isinstance(value, str) else set(value or [])
)

n_ingredients_median = float(raw_before['n_ingredients'].median())
n_steps_median = float(raw_before['n_steps'].median())
n_ingredients_iqr = float(raw_before['n_ingredients'].quantile(.75) - raw_before['n_ingredients'].quantile(.25)) or 1.0
n_steps_iqr = float(raw_before['n_steps'].quantile(.75) - raw_before['n_steps'].quantile(.25)) or 1.0
luxury_ingredients = {'lobster', 'truffle', 'wagyu', 'caviar', 'saffron', 'ribeye', 'shrimp', 'prosciutto'}
mod_pos_signals = {'asian': 1.8, 'mexican': 1.6, 'indian': 1.4, 'thai': 1.2, 'chinese': 1.2, 'middle-eastern': 1.2}
mod_neg_signals = {'north-american': -1.5, 'american': -1.0, 'comfort-food': -1.0}

def clamp01(value):
    return min(1.0, max(0.0, float(value)))

def raw_price_before(row):
    ingredients = {str(i).lower().strip() for i in row['ingredients']}
    tags = row['tags']
    price = 2.5 * clamp01(len(ingredients.intersection(luxury_ingredients)) / 2)
    if 'budget-friendly' in tags or 'cheap' in tags or '5-ingredients-or-less' in tags or row['n_ingredients'] <= 8:
        price -= 2.0 * clamp01((row['n_ingredients'] - 0) * 0 + (8 - row['n_ingredients']) / 7)
    return price

def raw_modification_before(row):
    tags = row['tags']
    modification = 0.0
    for tag, weight in mod_pos_signals.items():
        if tag in tags:
            modification += weight
    for tag, weight in mod_neg_signals.items():
        if tag in tags:
            modification += weight
    ingredient_complexity = (row['n_ingredients'] - n_ingredients_median) / n_ingredients_iqr
    step_complexity = (row['n_steps'] - n_steps_median) / n_steps_iqr
    modification += 0.6 * clamp01(ingredient_complexity / 2)
    modification += 0.6 * clamp01(step_complexity / 2)
    return modification

raw_before['price_raw_before'] = raw_before.apply(raw_price_before, axis=1)
raw_before['modification_raw_before'] = raw_before.apply(raw_modification_before, axis=1)

for column in ['price_raw_before', 'modification_raw_before']:
    counts = raw_before[column].value_counts().sort_values(ascending=False)
    mode_value = counts.index[0]
    mode_count = int(counts.iloc[0])
    print(f'{column}: mode={mode_value:.6g}, count={mode_count}, pct={mode_count / len(raw_before) * 100:.4f}%')

fig_price = px.histogram(raw_before, x='price_raw_before', nbins=80, title='Raw price before targeted fix')
fig_modification = px.histogram(raw_before, x='modification_raw_before', nbins=80, title='Raw modification before targeted fix')
fig_price.show()
fig_modification.show()

price_raw_before: mode=0, count=8823, pct=58.2607%
modification_raw_before: mode=0, count=3417, pct=22.5634%


Risposta Verifica A: il modo `price = 0.0` e il modo `modification = 0.0` sono default/assenza di contributi, non un singolo ramo attivo. Il valore specifico osservato sulle due ricette era invece spiegato da `price = -0.857143` prodotto dal gate `budget-friendly/cheap/5-ingredients-or-less/n_ingredients <= 8` e dalla penalita su `n_ingredients == 5`, e da `modification = -2.5` prodotto dalla somma discreta `north-american (-1.5) + american (-1.0)`.

## Verifica B - Distribuzione taste normalizzata

In [7]:
taste_stats = model.df_recipes['taste'].agg(['mean', 'median', 'max'])
print(f"mean={taste_stats['mean']:.4f}")
print(f"median={taste_stats['median']:.4f}")
print(f"p90={model.df_recipes['taste'].quantile(.9):.4f}")
print(f"p99={model.df_recipes['taste'].quantile(.99):.4f}")
print(f"max={taste_stats['max']:.4f}")
px.histogram(model.df_recipes, x='taste', nbins=80, title='Normalized taste after min-max').show()

mean=-0.9217
median=-0.7639
p90=0.9722
p99=2.6042
max=5.0000


## Fix mirato price/modification - Risultati dopo la modifica

Il fix rende `price` meno legato a soli conteggi discreti introducendo la comunanza media degli ingredienti, e rende i tag classici di `modification` dipendenti da una familiarita continua basata su ingredienti e step sotto mediana.

In [8]:
model_after_fix = MoodBasedRecommender()
model_after_fix.fit(df_recipes)
rng = np.random.default_rng(42)
idx = rng.choice(len(model_after_fix.df_mood_vectors), size=1000, replace=False)
rounded = np.round(model_after_fix.df_mood_vectors[idx], 2)
_, counts = np.unique(rounded, axis=0, return_counts=True)
print(f"duplicate_pairs_1000_before_targeted_fix = 0")
print(f"duplicate_pairs_1000_after_targeted_fix = {int(((counts * (counts - 1)) // 2).sum())}")
print(f"distanza_riferimento_after_targeted_fix = {model_after_fix.distanza_riferimento:.4f}")

-> Estrazione automatica delle 6 dimensioni del Mood per ogni ricetta...
-> Matrice Mood generata e normalizzata in scala [-5, +5]!
-> Distanza empirica di riferimento (p90): 8.8248
duplicate_pairs_1000_before_targeted_fix = 0
duplicate_pairs_1000_after_targeted_fix = 0
distanza_riferimento_after_targeted_fix = 8.8248


## Fix north-american / american

`american` e un sottoinsieme stretto di `north-american` nel dataset: il contributo `american` separato e stato rimosso per evitare doppio conteggio.

In [9]:
model_after_na_fix = MoodBasedRecommender()
model_after_na_fix.fit(df_recipes)
mod = model_after_na_fix.df_recipes['modification']
print(f"distanza_riferimento_p90 = {model_after_na_fix.distanza_riferimento:.4f}")
print(f"modification_mean = {mod.mean():.4f}")
print(f"modification_median = {mod.median():.4f}")
print(f"modification_min = {mod.min():.4f}")
print(f"modification_max = {mod.max():.4f}")
rng = np.random.default_rng(42)
idx = rng.choice(len(model_after_na_fix.df_mood_vectors), size=1000, replace=False)
rounded = np.round(model_after_na_fix.df_mood_vectors[idx], 2)
_, counts = np.unique(rounded, axis=0, return_counts=True)
print(f"duplicate_pairs_1000 = {int(((counts * (counts - 1)) // 2).sum())}")

-> Estrazione automatica delle 6 dimensioni del Mood per ogni ricetta...
-> Matrice Mood generata e normalizzata in scala [-5, +5]!
-> Distanza empirica di riferimento (p90): 8.8545
distanza_riferimento_p90 = 8.8545
modification_mean = -1.8803
modification_median = -1.7139
modification_min = -5.0000
modification_max = 5.0000
duplicate_pairs_1000 = 0


Altre inclusioni strette al 100% tra i tag attivi: `beef -> meat`, `pork -> meat`, `chinese -> asian`, `thai -> asian`. Nessun fix applicato a queste coppie.